# Frameflow — LTX-2.5 Video Studio on Colab

This notebook installs Wan2GP, starts the FastAPI server and HTML studio, automatically downloads/warms **LTX-2.5 Distilled 22B**, and opens a public Cloudflare URL.

It is configured for a **high-VRAM, high-system-RAM runtime** with WanGP **profile 1**. In Colab choose `Runtime → Change runtime type`, select the largest GPU available, and enable a high-RAM runtime before using `Runtime → Run all`. Profile 1 requires at least 64 GB of system RAM and 24 GB of VRAM; the notebook checks both before starting the server.

## 1. Verify the GPU

In [ ]:
import os, subprocess

try:
    subprocess.run(['nvidia-smi'], check=True)
except Exception as exc:
    raise RuntimeError(
        'GPU not detected. Open Runtime → Change runtime type, select GPU, save, and rerun.'
    ) from exc

gpu_line = subprocess.check_output([
    'nvidia-smi', '--query-gpu=name,memory.total',
    '--format=csv,noheader,nounits'
], text=True).strip().splitlines()[0]
GPU_NAME, gpu_memory_mib = [part.strip() for part in gpu_line.rsplit(',', 1)]
GPU_VRAM_GB = float(gpu_memory_mib) / 1024
SYSTEM_RAM_GB = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1024**3
print(f'GPU: {GPU_NAME} · {GPU_VRAM_GB:.1f} GiB VRAM')
print(f'System RAM: {SYSTEM_RAM_GB:.1f} GiB')
if GPU_VRAM_GB < 24 or SYSTEM_RAM_GB < 60:
    print('⚠ This runtime does not meet WanGP profile 1 minimums. Step 6 will stop with instructions.')

## 2. Download or update Wan2GP

In [ ]:
from pathlib import Path
import subprocess

WAN2GP_ROOT = Path('/content/wan2gp').resolve()
REPOSITORY = 'https://github.com/hoangthvn2201/wan2gp-optimized.git'
BRANCH = 'main'

if (WAN2GP_ROOT / '.git').exists():
    subprocess.run(['git', '-C', str(WAN2GP_ROOT), 'fetch', 'origin'], check=True)
else:
    subprocess.run(['git', 'clone', REPOSITORY, str(WAN2GP_ROOT)], check=True)
subprocess.run(['git', '-C', str(WAN2GP_ROOT), 'checkout', BRANCH], check=True)
subprocess.run(['git', '-C', str(WAN2GP_ROOT), 'pull', '--ff-only', 'origin', BRANCH], check=True)

required = [
    WAN2GP_ROOT / 'wan2gp_server' / '__main__.py',
    WAN2GP_ROOT / 'wan2gp_server' / 'static' / 'index.html',
    WAN2GP_ROOT / 'defaults' / 'ltx2_25_22B_distilled.json',
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise RuntimeError(f'The selected branch is missing Frameflow/LTX-2.5 files: {missing}')
print(f'Repository ready at {WAN2GP_ROOT}')

## 3. Install system libraries

In [ ]:
import os, subprocess

install_env = os.environ.copy()
install_env['DEBIAN_FRONTEND'] = 'noninteractive'
subprocess.run(['sudo', 'apt-get', 'update', '-qq'], check=True, env=install_env)
subprocess.run([
    'sudo', 'apt-get', 'install', '-y', '--no-install-recommends',
    'ffmpeg', 'libglib2.0-0', 'libgl1', 'libportaudio2'
], check=True, env=install_env)

## 4. Install Python dependencies

In [ ]:
import os, subprocess, sys

install_env = os.environ.copy()
install_env.setdefault('DEBIAN_FRONTEND', 'noninteractive')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip', 'setuptools', 'wheel'], check=True, env=install_env)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--force-reinstall', '--no-deps',
    'torch==2.7.1', 'torchvision==0.22.1', 'torchaudio==2.7.1',
    '--index-url', 'https://download.pytorch.org/whl/cu128'
], check=True, env=install_env)
# Frameflow uses SDPA. Remove any Colab-preinstalled xformers wheel that
# may target a different PyTorch ABI.
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'xformers'], check=False, env=install_env)
subprocess.run([
    sys.executable, '-m', 'pip', 'install',
    '-r', str(WAN2GP_ROOT / 'requirements.txt'),
    '-r', str(WAN2GP_ROOT / 'wan2gp_server' / 'requirements.txt')
], check=True, env=install_env)
print('Python environment ready.')

## 5. Apply the Colab headless compatibility setting

In [ ]:
target = WAN2GP_ROOT / 'preprocessing/matanyone/tools/interact_tools.py'
needle = "matplotlib.use('TkAgg')"
replacement = "matplotlib.use('Agg')"
if target.exists():
    source = target.read_text()
    if needle in source:
        target.write_text(source.replace(needle, replacement, 1))
        print('Enabled the headless matplotlib backend.')
    else:
        print('Headless backend already configured or no patch is needed.')

## 6. Configure the high-VRAM LTX-2.5 server

All three generation defaults point at the same LTX-2.5 Distilled checkpoint. Profile 1 keeps the active model in VRAM and reserves inactive weights in system RAM. The UI exposes Draft, High, and Max canvases up to approximately 1080p and clips up to 10 seconds. Startup stops early with a useful message when Colab has not provided enough RAM or VRAM.

In [ ]:
MEMORY_PROFILE = 1
PROFILE1_MIN_RAM_GB = 60   # nominal 64 GB runtimes report about 60–63 GiB
PROFILE1_MIN_VRAM_GB = 24
if SYSTEM_RAM_GB < PROFILE1_MIN_RAM_GB or GPU_VRAM_GB < PROFILE1_MIN_VRAM_GB:
    raise RuntimeError(
        f'WanGP profile 1 cannot safely start on this runtime: {SYSTEM_RAM_GB:.1f} GiB RAM / '
        f'{GPU_VRAM_GB:.1f} GiB VRAM detected. It needs a Colab high-RAM runtime '
        f'(about 64 GB RAM) and at least 24 GB VRAM. Change the runtime and run all cells again.'
    )

PORT = 8000
OUTPUT_DIR = Path('/content/frameflow_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SERVER_ENV = {
    'WAN2GP_ROOT': str(WAN2GP_ROOT),
    'WAN2GP_SERVER_PORT': str(PORT),
    'WAN2GP_SERVER_OUTPUT_DIR': str(OUTPUT_DIR),
    'WAN2GP_CLI_ARGS': f'--profile {MEMORY_PROFILE} --attention sdpa',
    'WAN2GP_SERVER_T2I_MODEL': 'ltx25-distilled-image',
    'WAN2GP_SERVER_T2V_MODEL': 'ltx25-distilled',
    'WAN2GP_SERVER_I2V_MODEL': 'ltx25-distilled-i2v',
    # The warmup below initializes the runtime exactly once. Eager init is
    # intentionally disabled to avoid racing it from a background thread.
    'WAN2GP_SERVER_EAGER_INIT': '0',
}
print('Profile: 1 (entire active model in VRAM)')
print('Default checkpoint: LTX-2.5 Distilled 22B')

## 7. Start Frameflow and automatically download LTX-2.5

This cell starts FastAPI, waits for it to become healthy, then submits one LTX-2.5 warmup. The first run downloads the checkpoint and supporting model files; later runs can reuse Colab's cache while the runtime remains alive. The cell monitors the actual server process while polling. If Colab kills it because of RAM/VRAM pressure, the notebook prints the process exit code, current resources, and the useful end of the server log instead of a Requests connection-reset traceback.

In [ ]:
import os, shutil, subprocess, sys, time
import requests

BASE_URL = f'http://127.0.0.1:{PORT}'
LOG_PATH = Path('/content/frameflow_server.log')
WARMUP_TIMEOUT = 4 * 3600

def log_tail(characters=14000):
    try:
        return LOG_PATH.read_text(errors='replace')[-characters:]
    except OSError as exc:
        return f'<could not read log: {exc}>'

def print_resource_snapshot():
    print('\n--- resource snapshot ---')
    subprocess.run([
        'nvidia-smi', '--query-gpu=name,memory.used,memory.total,utilization.gpu',
        '--format=csv,noheader'
    ], check=False)
    ram = {}
    for line in Path('/proc/meminfo').read_text().splitlines():
        key, value = line.split(':', 1)
        if key in {'MemTotal', 'MemAvailable', 'SwapTotal', 'SwapFree'}:
            ram[key] = int(value.strip().split()[0]) / 1024**2
    print(' · '.join(f'{key}={value:.1f} GiB' for key, value in ram.items()))
    free_disk = shutil.disk_usage('/content').free / 1024**3
    print(f'Free /content disk: {free_disk:.1f} GiB')

def fail_if_server_exited(stage):
    return_code = server_proc.poll()
    if return_code is None:
        return
    print_resource_snapshot()
    print(f'\n--- {LOG_PATH} (tail) ---')
    print(log_tail())
    if return_code in (-9, 137):
        reason = (
            'The OS killed the server (SIGKILL), which normally means profile 1 exhausted '
            'system RAM or VRAM. Use a Colab high-RAM runtime with a larger GPU.'
        )
    else:
        reason = 'The server exited; the log tail above contains the underlying error.'
    raise RuntimeError(
        f'Frameflow server exited during {stage} with code {return_code}. {reason}'
    )

# Always restart the process created by an earlier run of this cell so the
# current profile and dependency configuration is actually applied.
if 'server_proc' in globals() and server_proc.poll() is None:
    print(f'Restarting previous Frameflow server PID {server_proc.pid}…')
    server_proc.terminate()
    try:
        server_proc.wait(timeout=20)
    except subprocess.TimeoutExpired:
        server_proc.kill()
        server_proc.wait(timeout=10)

server_env = os.environ.copy()
server_env.update(SERVER_ENV)
server_env['PYTHONUNBUFFERED'] = '1'
server_log = open(LOG_PATH, 'w', buffering=1)
server_proc = subprocess.Popen(
    [sys.executable, '-u', '-m', 'wan2gp_server'],
    cwd=str(WAN2GP_ROOT), env=server_env,
    stdout=server_log, stderr=subprocess.STDOUT,
)
print(f'Server PID {server_proc.pid} · log {LOG_PATH}')

for _ in range(90):
    fail_if_server_exited('startup')
    try:
        health_response = requests.get(f'{BASE_URL}/health', timeout=2)
        if health_response.ok:
            print('Server online:', health_response.json())
            break
    except requests.RequestException:
        pass
    time.sleep(1)
else:
    print(log_tail())
    raise RuntimeError(f'Server did not start within 90 seconds. Inspect {LOG_PATH}.')

if str(WAN2GP_ROOT) not in sys.path:
    sys.path.insert(0, str(WAN2GP_ROOT))
from wan2gp_server.client import Wan2GPServerClient, Wan2GPServerError
client = Wan2GPServerClient(BASE_URL, timeout=120)

print('\nDownloading and warming LTX-2.5 Distilled. This can take several minutes…')
try:
    queued = client.preload(['ltx25-distilled'], wait=False)
except requests.RequestException as exc:
    fail_if_server_exited('warmup submission')
    raise RuntimeError(f'Could not submit the LTX-2.5 warmup: {exc}') from exc

if not queued:
    raise RuntimeError('The preload endpoint returned no warmup job.')
warmup_id = queued[0]['id']
print(f"--- preloading 'ltx25-distilled' (job {warmup_id}) ---")
deadline = time.monotonic() + WARMUP_TIMEOUT
last_line = ''
connection_failures = 0
warmup = queued[0]
while time.monotonic() < deadline:
    fail_if_server_exited('LTX-2.5 initialization')
    try:
        warmup = client.job(warmup_id)
        connection_failures = 0
    except requests.RequestException as exc:
        connection_failures += 1
        fail_if_server_exited('LTX-2.5 initialization')
        if connection_failures >= 12:
            print_resource_snapshot()
            print(log_tail())
            raise RuntimeError(
                f'Server stopped responding during model initialization: {exc}'
            ) from exc
        time.sleep(5)
        continue
    progress = warmup.get('progress') or {}
    line = (
        f"[{warmup['status']}] {progress.get('phase') or ''} "
        f"{progress.get('percent', 0)}% {progress.get('status') or ''}"
    ).strip()
    if line != last_line:
        print(line)
        last_line = line
    if warmup['status'] in {'succeeded', 'failed', 'cancelled'}:
        break
    time.sleep(5)
else:
    raise RuntimeError(f'LTX-2.5 warmup exceeded {WARMUP_TIMEOUT / 3600:.0f} hours.')

if warmup['status'] != 'succeeded':
    print(f'\n--- {LOG_PATH} (tail) ---')
    print(log_tail())
    raise RuntimeError(f"LTX-2.5 warmup {warmup['status']}: {warmup.get('error')}")
final_health = client.health()
if not final_health.get('runtime_loaded'):
    raise RuntimeError('Warmup succeeded but WanGP did not report a loaded runtime.')
print('✓ LTX-2.5 Distilled is downloaded, warm, and ready in VRAM.')

## 8. Expose the Studio and API

The tunnel publishes a clean Frameflow URL with no API key or URL fragment. Keep the Colab runtime alive while using the link.

In [ ]:
import os, queue, re, subprocess, threading
from IPython.display import HTML, display

CLOUDFLARED = Path('/content/cloudflared')
if not CLOUDFLARED.exists():
    subprocess.run([
        'wget', '-q', '-O', str(CLOUDFLARED),
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'
    ], check=True)
    subprocess.run(['chmod', '+x', str(CLOUDFLARED)], check=True)

if 'tunnel_proc' in globals() and tunnel_proc.poll() is None:
    tunnel_proc.terminate()

tunnel_proc = subprocess.Popen(
    [str(CLOUDFLARED), 'tunnel', '--url', BASE_URL, '--protocol', 'http2', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
url_queue = queue.Queue()
def drain_tunnel_output():
    for line in iter(tunnel_proc.stdout.readline, ''):
        match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
        if match and url_queue.empty():
            url_queue.put(match.group(0))
threading.Thread(target=drain_tunnel_output, daemon=True).start()

try:
    public_url = url_queue.get(timeout=90)
except queue.Empty as exc:
    tunnel_proc.terminate()
    raise RuntimeError('Cloudflare tunnel did not return a URL within 90 seconds.') from exc

studio_url = f'{public_url}/'
display(HTML(f'''
<div style="padding:20px 24px;border-radius:16px;background:#101d31;color:white;font-family:system-ui">
  <div style="font-size:12px;letter-spacing:.12em;color:#f26b51;font-weight:800">FRAMEFLOW IS READY</div>
  <a href="{studio_url}" target="_blank" style="display:inline-block;margin:12px 0 8px;padding:12px 18px;border-radius:10px;background:#f26b51;color:white;text-decoration:none;font-weight:700">Open Video Studio ↗</a>
  <div style="opacity:.65;font-size:12px">API docs: <a href="{public_url}/docs" target="_blank" style="color:#a9c6ef">{public_url}/docs</a></div>
</div>
'''))
print('Studio:', studio_url)

## 9. Operations and troubleshooting

- Server log: `/content/frameflow_server.log`
- Generated media: `/content/frameflow_outputs`
- Studio/API health: `http://127.0.0.1:8000/health`
- Stop services with the optional cell below.
- Profile 1 needs both high system RAM and high VRAM. A high-VRAM GPU alone is not sufficient; enable Colab's high-RAM runtime.
- Exit code `-9` during warmup means the OS killed the server, normally because RAM or VRAM was exhausted. The startup cell now prints resource and log diagnostics automatically.
- This notebook uses PyTorch 2.7.1 because WanGP warns that 2.8 can leak system RAM. Restart the Colab runtime before rerunning if an older version was already imported in the current Python process.
- If this runtime cannot meet profile 1 requirements, change `MEMORY_PROFILE` to `3.5` and lower the RAM check, or use the low-VRAM `wan2gp_server.ipynb`.
- A Cloudflare quick-tunnel URL is temporary and changes whenever the tunnel cell is rerun.

In [ ]:
# Optional: uncomment to stop both background services.
# if 'tunnel_proc' in globals() and tunnel_proc.poll() is None: tunnel_proc.terminate()
# if 'server_proc' in globals() and server_proc.poll() is None: server_proc.terminate()
# print('Frameflow services stopped.')